# 강의 03 · 실습 4 — RAG 에이전트 서비스 · (6) 고난도 III

## 1. 문제상황

- 구름월드 안내 서비스는 검색할지 말지를 모델이 정합니다. 시험 운영에서 사실 질문인데 모델이 검색을 건너뛰고 자기 지식으로 답한 일이 몇 번 있었습니다.
- 시스템 프롬프트로 「반드시 검색하라」를 넣어도 모델의 판단이라 100%가 되지 않습니다. 근거 없음 가드는 검색을 한 번이라도 했을 때만 작동합니다.
- 담당자는 검색 시점을 정하는 다른 구조, 즉 질문의 성격과 무관하게 코드가 검색을 먼저 실행해 그 결과를 프롬프트에 넣는 구조를 시험해 보려 합니다.
- 다만 인사말까지 검색하면 낭비이므로, 인사말인지 사실 질문인지를 먼저 두 값 중 하나로 판정해 사실 질문에만 무조건 검색을 적용하고, 두 구조의 검색 횟수와 답을 같은 질문으로 견주어 보고 싶어 합니다.

## 2. 문제와 목표

- **문제**: 모델이 검색을 건너뛰는 경로를 시스템 프롬프트만으로는 막지 못하고, 근거 없음 가드는 검색 0회 경로에 작동하지 않습니다.
- **목표**: 질문을 인사말·사실 질문 두 값으로 판정하는 구조화 출력 판정기를 두고, 사실 질문이면 코드가 검색을 먼저 실행해 근거(또는 검색 결과 없음)를 프롬프트에 넣어 답하게 하는 「무조건 검색」 처리 함수를 만들어, 「판단해 검색」 처리 함수(모델이 검색 시점을 정하는 손 루프)와 같은 질문으로 검색 횟수와 답을 견주고, 무조건 검색 쪽을 서비스 주소로 노출합니다.
    - 판정기: 질문을 인사말·사실 질문 두 값 중 하나로 판정하는 구조화 출력.
    - 무조건 검색: 사실 질문이면 코드가 검색 도구를 먼저 실행하고, 결과가 없으면 모델을 부르지 않고 고정 안내 문장을 돌려줍니다.
    - 판단해 검색: 모델이 도구를 부를지 정하는 손 루프(비교 대상).
    - 비교표: 질문 · 구조 · 검색 횟수 · 답 앞부분.
    - 주어진 것: 데이터 파일 `day05_faq_구름월드.csv`(32행, 행마다 `[카테고리] Q: … A: …` 본문과 행 번호·카테고리 메타데이터), 임베딩 모델 `text-embedding-3-small`, 저장소 디렉터리 `chroma_db`.
    - 시스템 프롬프트와 고정 안내 문장은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**: 인사말은 두 구조 모두 검색 0회로 즉답하고, 환불 질문은 두 구조 모두 근거로 답하며, 문서 밖 질문은 무조건 검색 구조에서 검색 1회 뒤 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」로 끝나는 것을 비교표와 서비스 호출에서 확인합니다.
    - 검색은 상위 2개 청크와 거리 점수, 임계값은 1.5입니다.
    - 서비스 주소는 `http://127.0.0.1:8031`(포트 8031)입니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리 불러오기, `.env` 읽기, 모델 준비는 주어진 것입니다. 아래 셀을 고치지 않고 그대로 실행합니다.

- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time

import httpx
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from typing import Literal

from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

# 주어진 자료
NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."
SYSTEM = ("너는 시설 안내 담당자다. 인사말처럼 검색이 필요 없는 말에는 바로 답한다. "
          "그 밖의 모든 질문은 반드시 faq_search 도구로 근거를 먼저 찾고, 도구 결과에 있는 내용으로만 답한다. "
          "도구 결과가 '검색 결과 없음'이면 네가 아는 지식으로 답하지 말고 "
          f"'{NO_EVIDENCE}'라고만 답한다.")

In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 판정기가 인사말과 사실 질문을 두 값으로 구분합니다.
2. 비교표에서 인사말은 두 구조 모두 검색 0회, 환불 질문은 두 구조 모두 근거로 답하며, 문서 밖 질문은 무조건 검색 구조에서 검색 1회 뒤 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」입니다. 판단해 검색 구조의 문서 밖 질문 검색 횟수는 모델의 판단이라 실행마다 다를 수 있습니다. 그 차이가 이 실습의 요점입니다.
3. 서비스 응답이 비교표의 무조건 검색 행과 같은 성격입니다.

세 가지가 모두 확인되면 완성입니다.